In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, root_mean_squared_error
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ==========================================
# 1. 데이터 로드
# ==========================================
features_df = pd.read_csv('/Users/kwagminseo/Documents/SAS/data/train_features_merged.csv')
info_df = pd.read_csv("/Users/kwagminseo/Documents/SAS/data/train/train_customer_info.csv")
targets_df = pd.read_csv("/Users/kwagminseo/Documents/SAS/data/train/train_targets.csv")
trans_df = pd.read_csv('/Users/kwagminseo/Documents/SAS/data/train/train_transaction_history.csv')  # 거래내역 (경로 확인 요망)

In [4]:
# ==========================================
# 2. ⭐️ 킬러 피처: 시계열 추세(Trend) 변수 생성
# ==========================================
print("▶️ 고객별 소비 추세(Trend) 분석 중...")
def extract_trend_features(df, base_date_str='2023-12-31'):
    df['trans_date'] = pd.to_datetime(df['trans_date'])
    base_date = pd.to_datetime(base_date_str)
    
    # 1) 전체 기간 요약
    agg_all = df.groupby('customer_id').agg(
        total_freq=('trans_id', 'count'),
        total_monetary=('trans_amount', 'sum')
    ).reset_index()
    
    # 2) 최근 90일(3개월) 요약
    recent_90d = base_date - pd.Timedelta(days=90)
    recent_df = df[df['trans_date'] >= recent_90d]
    agg_recent = recent_df.groupby('customer_id').agg(
        recent_3m_freq=('trans_id', 'count'),
        recent_3m_monetary=('trans_amount', 'sum')
    ).reset_index()
    
    # 3) 병합 및 결측치 처리
    trans_features = agg_all.merge(agg_recent, on='customer_id', how='left').fillna(0)
    
    # 4) 🔥 킬러 피처: 소비 하락/상승 비율 (과거 대비 최근)
    # (최근 3개월 평균) / (전체 기간 평균) 
    # 값이 1보다 작으면 최근에 소비가 줄어든 것 (이탈 위험 🚨)
    trans_features['spending_trend_ratio'] = (trans_features['recent_3m_monetary'] / 3) / ((trans_features['total_monetary'] + 1) / 12)
    trans_features['freq_trend_ratio'] = (trans_features['recent_3m_freq'] / 3) / ((trans_features['total_freq'] + 1) / 12)
    
    return trans_features

trans_agg = extract_trend_features(trans_df)

▶️ 고객별 소비 추세(Trend) 분석 중...


In [7]:
# ==========================================
# 3. 데이터 병합 및 재무/라이프 파생 변수
# ==========================================
print("▶️ 파생 변수 결합 및 데이터 정제 중...")
info_cols = ['customer_id', 'gender', 'region_code', 'is_married', 'prefer_category', 'income_group']
df = pd.merge(features_df, info_df[info_cols], on='customer_id', how='left')
df = pd.merge(df, trans_agg, on='customer_id', how='left')
df = pd.merge(df, targets_df[['customer_id', 'target_churn', 'target_ltv']], on='customer_id', how='left')

# 나이 및 자산 파생변수 (try1의 좋은 아이디어 채용)
df['age_group'] = pd.cut(df['age'], bins=[0, 29, 39, 49, 59, 100], labels=['20대', '30대', '40대', '50대', '60대'])
df['life_stage'] = df['age_group'].astype(str) + '_' + np.where(df['is_married'] == 1, '기혼', '미혼')
df['net_worth'] = df['total_deposit_balance'] - df['total_loan_balance']
df['burn_rate'] = df['total_spending'] / (df['total_deposit_balance'] + 1) # 자산 소진율
df['tenure_group'] = pd.qcut(df['tenure_days'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

▶️ 파생 변수 결합 및 데이터 정제 중...


In [8]:
# ==========================================
# 4. 인코딩: LightGBM 네이티브 모드 (category 타입 지정)
# ==========================================
cat_cols = ['gender', 'region_code', 'prefer_category', 'income_group', 'age_group', 'life_stage', 'tenure_group']

# 원핫, 라벨 인코딩 다 버리고 판다스 category 타입으로만 바꿉니다.
for col in cat_cols:
    df[col] = df[col].astype('category')

df.replace([np.inf, -np.inf], np.nan, inplace=True)

,customer_id,age,income_num,tenure_days,credit_score,total_deposit_balance,total_loan_balance,fin_overdue_days,fin_asset_trend_score,total_spending,...,recent_3m_monetary,spending_trend_ratio,freq_trend_ratio,target_churn,target_ltv,age_group,life_stage,net_worth,burn_rate,tenure_group
0,C000001,36,4,1359,713,2517740,58608891,0,0.551107,1087476,...,898628.0,3.305368,2.363636,0,556691.00,30대,30대_기혼,-56091151,0.431925,Q4
1,C000002,32,3,1026,869,679696,33403843,0,-0.342776,922154,...,391661.0,1.698894,2.133333,0,1460203.00,30대,30대_미혼,-32724147,1.356713,Q3
2,C000003,41,3,601,588,17319511,0,1,-0.949003,812618,...,431224.0,2.122638,2.352941,0,605476.00,40대,40대_기혼,17319511,0.046919,Q1
3,C000004,23,3,1191,742,984771,0,0,-0.348409,1237060,...,838753.0,2.712083,2.086957,0,1034150.00,20대,20대_기혼,984771,1.256189,Q3
4,C000006,31,3,1390,611,1098491,1668819,0,0.647922,1034043,...,291865.0,1.129024,1.333333,1,76083.15,30대,30대_미혼,-570328,0.941330,Q4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,C099993,23,4,1085,616,827808595,11991094,0,-2.387639,1521949,...,1165059.0,3.062016,2.833333,0,2842492.00,20대,20대_기혼,815817501,0.001839,Q3
59996,C099995,34,5,1274,584,5095394,32205717,1,0.847797,1318712,...,613000.0,1.859389,1.666667,0,541730.00,30대,30대_미혼,-27110323,0.258805,Q4
59997,C099996,19,5,812,847,1272279,0,0,-0.162601,1211965,...,435470.0,1.437235,1.555556,0,593719.00,20대,20대_미혼,1272279,0.952593,Q2
59998,C099998,33,3,598,899,32067940,0,0,-0.312655,1472751,...,981999.0,2.667113,2.095238,0,3720969.00,30대,30대_기혼,32067940,0.045926,Q1


In [9]:
# ==========================================
# 5. 검증셋 분할 (Raw LTV 사용)
# ==========================================
X = df.drop(columns=['customer_id', 'target_churn', 'target_ltv'])
y_churn = df['target_churn']
y_ltv = df['target_ltv']

X_train, X_val, y_churn_train, y_churn_val, y_ltv_train, y_ltv_val = train_test_split(
    X, y_churn, y_ltv, test_size=0.2, random_state=42, stratify=y_churn
)

In [14]:
# ==========================================
# 6. 고성능 LightGBM 학습
# ==========================================
print("▶️ [STAGE 1] 불균형을 고려한 이탈 모델(LGBM) 학습 중...")
# scale_pos_weight: 이탈(1) 데이터가 10%라면 약 9배의 가중치를 줘서 더 깐깐하게 학습시킴
churn_ratio = (len(y_churn_train) - sum(y_churn_train)) / sum(y_churn_train)

clf = lgb.LGBMClassifier(
    random_state=42, 
    n_estimators=500,        # 트리 개수 넉넉히
    learning_rate=0.01,      # 세밀하게 학습
    scale_pos_weight=churn_ratio, # 🔥 불균형 타파
    verbose=-1
)
clf.fit(X_train, y_churn_train, eval_set=[(X_val, y_churn_val)], callbacks=[lgb.early_stopping(50)])
churn_val_pred = clf.predict_proba(X_val)[:, 1]
auc_score = roc_auc_score(y_churn_val, churn_val_pred)
print(f"✅ 이탈 모델 AUC Score: {auc_score:.4f}")

print("\n▶️ [STAGE 2] 이상치(VVIP)에 대응하는 LTV 예측 모델(LGBM) 학습 중...")
reg = lgb.LGBMRegressor(
    random_state=42, 
    n_estimators=500, 
    learning_rate=0.01, 
    verbose=-1
)
reg.fit(X_train, y_ltv_train, eval_set=[(X_val, y_ltv_val)], callbacks=[lgb.early_stopping(50)])
ltv_val_pred = np.maximum(reg.predict(X_val), 0)
rmse = root_mean_squared_error(y_ltv_val, ltv_val_pred)
print(f"✅ LTV 모델 RMSE: {rmse:,.0f} 원")

▶️ [STAGE 1] 불균형을 고려한 이탈 모델(LGBM) 학습 중...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[14]	valid_0's binary_logloss: 0.308761
✅ 이탈 모델 AUC Score: 0.7731

▶️ [STAGE 2] 이상치(VVIP)에 대응하는 LTV 예측 모델(LGBM) 학습 중...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[255]	valid_0's l2: 1.87218e+12
✅ LTV 모델 RMSE: 1,368,278 원


In [15]:
# ==========================================
# 7. 최종 예상 스코어
# ==========================================
final_score = 0.5 * auc_score + 0.5 * (1 / (1 + np.log(rmse)))
print("\n" + "="*50)
print(f"🏆 예상 최종 통합 점수: {final_score:.4f}")
print("="*50)


🏆 예상 최종 통합 점수: 0.4196
